# MalNetTiny dataset showcase

This notebook loads PyG's MalNetTiny dataset through `dataset/MalNet.py`, prints split statistics, inspects a few graph examples, and checks the dataloaders.

In [7]:
from collections import Counter
from pathlib import Path
import sys

import torch

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from dataset.MalNetTiny import (
    _load_malnet_tiny,
    build_malnettiny_dataloaders,
    build_malnettiny_pyg_dataloaders,
)

DATA_ROOT = REPO_ROOT / "data" / "malnet_tiny"

## Load splits

In [8]:
train_ds = _load_malnet_tiny(str(DATA_ROOT), split="train")
val_ds = _load_malnet_tiny(str(DATA_ROOT), split="val")
test_ds = _load_malnet_tiny(str(DATA_ROOT), split="test")
full_ds = _load_malnet_tiny(str(DATA_ROOT), split=None)

print(f"Full dataset: {len(full_ds):,} graphs")
print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")

Full dataset: 5,000 graphs
Train: 3,500 | Val: 500 | Test: 1,000


## Dataset stats

In [9]:
def summarize_split(name, dataset):
    node_counts = torch.tensor([int(g.num_nodes) for g in dataset], dtype=torch.float)
    edge_counts = torch.tensor(
        [int(g.edge_index.shape[1]) for g in dataset], dtype=torch.float
    )
    labels = [int(g.y.item()) if hasattr(g.y, "item") else int(g.y) for g in dataset]
    label_counts = Counter(labels)

    print(f"\n{name}")
    print(f"  graphs: {len(dataset):,}")
    print(
        "  nodes: "
        f"min={int(node_counts.min())}, "
        f"mean={node_counts.mean().item():.1f}, "
        f"median={node_counts.median().item():.1f}, "
        f"max={int(node_counts.max())}"
    )
    print(
        "  edges: "
        f"min={int(edge_counts.min())}, "
        f"mean={edge_counts.mean().item():.1f}, "
        f"median={edge_counts.median().item():.1f}, "
        f"max={int(edge_counts.max())}"
    )
    print(f"  labels: {dict(sorted(label_counts.items()))}")


for split_name, split_ds in [
    ("train", train_ds),
    ("val", val_ds),
    ("test", test_ds),
    ("full", full_ds),
]:
    summarize_split(split_name, split_ds)


train
  graphs: 3,500
  nodes: min=5, mean=1504.7, median=987.0, max=14166
  edges: min=4, mean=2831.1, median=1640.0, max=20096
  labels: {0: 700, 1: 700, 2: 700, 3: 700, 4: 700}

val
  graphs: 500
  nodes: min=13, mean=1527.3, median=849.0, max=5415
  edges: min=7, mean=2867.5, median=1396.0, max=19916
  labels: {0: 100, 1: 100, 2: 100, 3: 100, 4: 100}

test
  graphs: 1,000
  nodes: min=6, mean=1580.3, median=1036.0, max=14166
  edges: min=5, mean=2957.3, median=1715.0, max=14192
  labels: {0: 200, 1: 200, 2: 200, 3: 200, 4: 200}

full
  graphs: 5,000
  nodes: min=5, mean=1522.1, median=994.0, max=14166
  edges: min=4, mean=2859.9, median=1640.0, max=20096
  labels: {0: 1000, 1: 1000, 2: 1000, 3: 1000, 4: 1000}


## Example graphs

In [10]:
for idx in range(5):
    graph = train_ds[idx]
    label = int(graph.y.item()) if hasattr(graph.y, "item") else int(graph.y)
    print(f"Example {idx}")
    print(f"  label: {label}")
    print(f"  num_nodes: {graph.num_nodes}")
    print(f"  num_edges: {graph.edge_index.shape[1]}")
    print(f"  first 10 edges:\n{graph.edge_index[:, :10]}")

Example 0
  label: 0
  num_nodes: 1679
  num_edges: 3576
  first 10 edges:
tensor([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10]])
Example 1
  label: 0
  num_nodes: 3520
  num_edges: 8838
  first 10 edges:
tensor([[  1,   2,   4,   6,   8,   8,  10,  10,  10,  10],
        [  2,   3,   5,   7,   6,   9,   9,  60, 414,  13]])
Example 2
  label: 1
  num_nodes: 221
  num_edges: 351
  first 10 edges:
tensor([[  0,   2,   2,   2,   2,   2,   2,   2,   2,   2],
        [  1,   0,   6,   8, 118, 119, 120, 121, 122, 123]])
Example 3
  label: 1
  num_nodes: 3001
  num_edges: 4785
  first 10 edges:
tensor([[  0,   2,   2,   2,   2,   2,   2,   2,   2,   2],
        [  1,   0,   6,   8, 120, 121, 122, 123, 124, 125]])
Example 4
  label: 2
  num_nodes: 122
  num_edges: 57
  first 10 edges:
tensor([[  1,   3,   3,   3,   7,   7,   8,   8,   8,   8],
        [  2,   4,   5,   6,   3, 121,   9,  10,  11,  12]])


## Raw PyG dataloader

In [11]:
train_loader, test_loader = build_malnettiny_pyg_dataloaders(
    root=str(DATA_ROOT),
    batch_size=4,
    num_workers=0,
)

batch = next(iter(train_loader))
print(batch)
print(f"batch graphs: {batch.num_graphs}")
print(f"batched nodes: {batch.num_nodes}")
print(f"batched edges: {batch.edge_index.shape[1]}")
print(f"labels: {batch.y.tolist()}")

DataBatch(edge_index=[2, 7634], y=[4], num_nodes=4500, batch=[4500], ptr=[5])
batch graphs: 4
batched nodes: 4500
batched edges: 7634
labels: [3, 3, 4, 3]


## Graph-text dataloader smoke test

In [12]:
from graph_tokenization import AutoGraphTokenizer


class WhitespaceTokenizer:
    bos_token_id = 1
    eos_token_id = 2

    def __init__(self):
        self.vocab = {}

    def encode(self, text, add_special_tokens=False):
        ids = []
        for token in text.split():
            if token not in self.vocab:
                self.vocab[token] = len(self.vocab) + 10
            ids.append(self.vocab[token])
        return ids


graph_tokenizer = AutoGraphTokenizer(
    dataset_names=["MalNetTiny"],
    max_length=128,
    undirected=True,
    append_eos=True,
)
text_tokenizer = WhitespaceTokenizer()

text_train_loader, text_test_loader = build_malnettiny_dataloaders(
    root=str(DATA_ROOT),
    graph_tokenizer=graph_tokenizer,
    llada_tokenizer=text_tokenizer,
    num_workers=0,
)

text_batch = next(iter(text_train_loader))
print(text_batch.keys())
print(f"input_ids shape: {tuple(text_batch['input_ids'].shape)}")
print(f"prompt_lengths: {text_batch['prompt_lengths'].tolist()}")
print(f"graph_indices: {text_batch['graph_indices']}")
print(f"num_nodes: {text_batch['num_nodes'].tolist()}")
print(f"num_edges: {text_batch['num_edges'].tolist()}")
print(f"labels: {[int(y.item()) for y in text_batch['labels']]}")

[MalNetTiny] Loaded splits: train=4000, test=1000.
[MalNetTiny] Graph tokenizer vocab set for max 14166 nodes.
dict_keys(['input_ids', 'prompt_lengths', 'graph_indices', 'labels', 'num_nodes', 'num_edges'])
input_ids shape: (1, 135)
prompt_lengths: [4]
graph_indices: [2587]
num_nodes: [2293]
num_edges: [4933]
labels: [4]
